# 第 8 章 ナイーブベイズ

学習ループのない学習器です。単語を数え上げ、ラプラス平滑化を掛け、対数空間で確率を合成します。

対応する記事: [第 8 章 ナイーブベイズ（F# 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/fsharp/ch08.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch08NaiveBayes.fs"

open GrokkingMl.Ch08NaiveBayes

## データセット

スパム 3 通、通常メール 5 通の小さなデータです。**学習は数え上げの 1 パスで終わります。**

In [2]:
let documents =
    [ "lottery sale"; "lottery winning"; "winning lottery sale"; "sale today"
      "meeting tomorrow"; "project meeting"; "lunch meeting today"; "project deadline" ]

let labels = [ 1; 1; 1; 0; 0; 0; 0; 0 ]

let model = train documents labels
printfn "スパム %d 通 / 通常 %d 通" model.SpamDocuments model.HamDocuments
printfn "事前確率 %.4f" (priorSpamProbability model)

スパム 

3

 通 / 通常 

5

 通

事前確率 

0.3750

## ラプラス平滑化

`lottery` はスパム 3 件・通常 0 件です。**平滑化がないと確率 1.0 になり、他のどんな単語が来ても覆せません。** すべてのカウントに 1 を足すことで 0.8 に収まります。

未知語はちょうど 0.5 になり、**判定に寄与しません。**

In [3]:
let countIn counts word =
    Map.tryFind word counts |> Option.defaultValue 0

printfn "%-10s %6s %6s %8s" "単語" "スパム" "通常" "確率"

for word in [ "lottery"; "winning"; "sale"; "today"; "meeting"; "unseen" ] do
    printfn "%-10s %6d %6d %8.4f" word (countIn model.SpamWordCounts word)
        (countIn model.HamWordCounts word) (wordSpamProbability model word)

単語        

   スパム

    通常

      確率

lottery   

...

  0.2000

unseen    

     0

     0

  0.5000

## 証拠が積み重なる

スパム語が重なるほど確率が上がります。これが掛け算（実装上は対数の足し算）の効果です。

**空の文書は事前確率そのもの** を返します。単語による更新が 1 つも起きないためです。

In [4]:
for document in [ ""; "project deadline"; "sale today"; "lottery"
                  "lottery winning"; "lottery winning sale" ] do
    let probability = predictProbability model document
    let bar = String.replicate (int (probability * 40.0)) "#"
    let name = if document = "" then "(空)" else document
    printfn "%-24s %.4f %s" name probability bar

(空)                     

0.3750

###############

project deadline        

0.0909

...

0.8780

###################################

lottery winning sale    

0.9153

####################################

## 未知語は予測を変えない

平滑化により未知語の確率は 0.5 なので、含めても含めなくても結果は同じです。

In [5]:
printfn "lottery              %.6f" (predictProbability model "lottery")
printfn "lottery zzzz qqqq    %.6f" (predictProbability model "lottery zzzz qqqq")
printfn "正解率 %.2f" (accuracy model documents labels)

lottery              

0.705882

lottery zzzz qqqq    

0.705882

正解率 

1.00

## 試してみる: 学習データを足す

新しいメールを 1 通足すと、単語の確率がどう動くでしょうか。**学習ループがないので、足して数え直すだけです。**

In [6]:
let extended = train (documents @ [ "meeting lottery" ]) (labels @ [ 0 ])

printfn "%-10s %8s %8s" "単語" "元" "追加後"

for word in [ "lottery"; "meeting" ] do
    printfn "%-10s %8.4f %8.4f" word (wordSpamProbability model word) (wordSpamProbability extended word)

単語        

       元

     追加後

lottery   

  0.8000

  0.6667

meeting   

  0.2000

  0.1667